### California Place Rate Statistics
This notebook gets all census places in California

Then calculates their population percentages per each race, per white, and per persons of color, for ACS 5 year 2020-2024

Mapping Police Violence uses the following race categories: Hispanic, White, Black, Unknown race, Asian, Native Hawaiian and Pacific Islander, and American Indian and Alaska Native so those categories will be mapped to Census categories

In [32]:
import pandas as pd
from census import Census
import numpy as np

# Census API access
api_key = "" # enter in your own key here
c = Census(key=api_key)

In [45]:
def combine_cols(in_df, combined_col_name, combine_cols=None, out_cols=None):
    '''
    Inputs:
    - in_df (pd.DataFrame): a DataFrame 
    - cols (list): columns to combine
    - combined_col_name (string): output column to produce

    Outputs:
    A modified version of in_df with certain columns grouped and proportions and margins of error calculated
    '''
    
    df = in_df.copy()
    df = df.replace(-555555555.0, 0)

    if combine_cols is not None:
        # Calculate combined
        df[combined_col_name] = df[combine_cols].sum(axis='columns')
        combined_moes = [f'{col}_moe' for col in combine_cols]
        df[f'{combined_col_name}_moe'] = (df[combined_moes]**2).sum(axis='columns')**0.5

    ### CALCULATE PROPORTION
    # Calculate the proportion for all cols being used
    for group in out_cols:
        df[f'{group}_pct'] = df[group] / df['total_pop']
    
        # Calculate the MOE for this proportion
        df[f'pct_{group}_moe'] = (df[f'{group}_moe']**2 - df[f'{group}_pct']**2 * df['total_pop_moe']**2)**0.5 / df['total_pop']
    
        #NaN-out any tracts of too-low absolute n
        df.loc[df.total_pop < 25, f'{group}_pct'] = float('NaN')
        df.loc[df.total_pop < 25, f'pct_{group}_moe'] = float('NaN')
        
        #NaN-out any tracts of too-low moe
        df[f'pct_{group}_moe_ratio'] = df[f'pct_{group}_moe']/df[f'{group}_pct']
        df.loc[df[f'pct_{group}_moe_ratio'] > .4, f'{group}_pct'] = float('NaN')
        df.loc[df[f'pct_{group}_moe_ratio'] > .4, f'pct_{group}_moe'] = float('NaN')
    
    return df

In [42]:
race_in_variables = {
    'B03002_001E': 'total_pop',
    'B03002_001M': 'total_pop_moe',
    'B03002_003E': 'white_pop',
    'B03002_003M': 'white_pop_moe',
    'B03002_004E': 'black_pop',
    'B03002_004M': 'black_pop_moe',
    'B03002_005E': 'native_pop',
    'B03002_005M': 'native_pop_moe',
    'B03002_006E': 'asian_pop',
    'B03002_006M': 'asian_pop_moe',
    'B03002_007E': 'pi_pop',
    'B03002_007M': 'pi_pop_moe',
    'B03002_008E': '1other_pop',
    'B03002_008M': '1other_pop_moe',
    'B03002_009E': 'multi_pop',
    'B03002_009M': 'multi_pop_moe',
    'B03002_012E': 'hispanic_pop',
    'B03002_012M': 'hispanic_pop_moe',
}
race_other_cols = ['1other_pop', 'multi_pop']
race_other_out_cols = ['unknown_pop', 'white_pop', 'native_pop', 'pi_pop', 'black_pop', 'asian_pop', 'hispanic_pop']

race_poc_cols = ['1other_pop', 'multi_pop', 'native_pop', 'pi_pop', 'black_pop', 'asian_pop', 'hispanic_pop']
race_poc_out_cols = ['white_pop', 'poc_pop', 'unknown_pop', 'native_pop', 'pi_pop', 'black_pop', 'asian_pop', 'hispanic_pop']

# Columns to keep for export
race_keep_cols = [
    'total_pop',
    'place',
    'white_pop',  
    'native_pop', 
    'pi_pop', 
    'black_pop', 
    'asian_pop', 
    'hispanic_pop',
    'unknown_pop',
    'poc_pop', 
    'white_pop_pct',
    'native_pop_pct',
    'pi_pop_pct',
    'black_pop_pct',
    'asian_pop_pct',
    'hispanic_pop_pct',
    'unknown_pop_pct',
    'poc_pop_pct',
]

In [43]:
# Get ACS 2024 5 year Table B03002 for all places in California
place_race = pd.DataFrame(
    c.acs5.get(
        list(race_in_variables.keys()),
        {'for': 'place:*', 'in': 'state:06'},
        year=2024
    )
)
place_race = place_race.rename(columns=race_in_variables)

In [46]:
# First, combine multi and other into "Unknown" to match MPV
place_race_unknown = combine_cols(place_race, "unknown_pop", race_other_cols, race_other_out_cols)

In [47]:
# Then combine all non-white races to add a poc column
place_race_poc = combine_cols(place_race_unknown, "poc_pop", race_poc_cols, race_poc_out_cols)

In [48]:
place_race_poc = place_race_poc[race_keep_cols]

In [49]:
# Match census column name for place
place_race_poc = place_race_poc.rename(columns={"place":"PLACEFP"})

In [51]:
place_race_poc.to_csv("data/place_populations.csv", index=False)